In [5]:
# @title x-sel with SHAP

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import scipy.stats as stats

from sklearn.base import clone
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

import xgboost as xgb
from statsmodels.stats.outliers_influence import variance_inflation_factor
from tabulate import tabulate

# -----------------------
# CONFIG
# -----------------------
TRAIN_PATH = "/content/Case 3 Train.xlsx"   # change if needed
TEST_PATH  = "/content/Case 3 Test.xlsx"    # change if needed
TARGET = 'D_ext'
DROP_COLS = ['Time(s)', 'TI', 'Time (s)']  # columns to ignore when forming features
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
N_FOLDS = 5
DO_GRIDSEARCH = True   # set False to skip GridSearch
GRID_JOBS = -1

# -----------------------
# Utilities & metrics
# -----------------------
def rmsre(y_true, y_pred):
    with np.errstate(divide='ignore', invalid='ignore'):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        rel_err = np.where(y_true != 0, (y_true - y_pred) / y_true, 0.0)
        return float(np.sqrt(np.mean(rel_err ** 2)))

def mape(y_true, y_pred):
    with np.errstate(divide='ignore', invalid='ignore'):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        ape = np.where(y_true != 0, np.abs((y_true - y_pred)/y_true), 0.0)
        return float(np.mean(ape) * 100.0)

def save_fig(fig, filename, dpi=500):
    fig.savefig(filename, dpi=dpi, bbox_inches='tight')
    plt.close(fig)

# -----------------------
# Load data
# -----------------------
print("Loading data...")
train_df = pd.read_excel(TRAIN_PATH).dropna().reset_index(drop=True)
test_df  = pd.read_excel(TEST_PATH).dropna().reset_index(drop=True)

features = [c for c in train_df.columns if c not in ([TARGET] + DROP_COLS)]
X_train = train_df[features].copy()
y_train = train_df[TARGET].copy()
X_test = test_df[features].copy()
y_test = test_df[TARGET].copy()

print(f"Features ({len(features)}): {features}")

# scale features
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# -----------------------
# Models + hyperparameter grids
# -----------------------
base_models = {
    'LinearRegression': (LinearRegression(), {'fit_intercept': [True, False], 'positive': [True, False]}),
    'DecisionTree': (DecisionTreeRegressor(random_state=RANDOM_SEED), {'max_depth': [None, 10, 20], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]}),
    'RandomForest': (RandomForestRegressor(random_state=RANDOM_SEED), {'n_estimators': [100, 200], 'max_depth': [None, 10], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]}),
    'GradientBoosting': (GradientBoostingRegressor(random_state=RANDOM_SEED), {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.1], 'max_depth': [3, 5]}),
    'SVR': (SVR(), {'C': [0.1, 1, 10], 'gamma': ['scale'], 'kernel': ['linear', 'rbf']}),
    'kNN': (KNeighborsRegressor(), {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']}),
    'XGBoost': (xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_SEED), {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.1], 'max_depth': [3, 6]})
}

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# containers
oof_preds = np.zeros((X_train_scaled.shape[0], len(base_models)))
test_preds = np.zeros((X_test_scaled.shape[0], len(base_models)))
base_results = []
best_params_base = {}
trained_base_models = {}

# -----------------------
# Training base models (GridSearchCV then OOF/test preds)
# -----------------------
print("\nTraining base models...")
for i, (name, (model_cls, param_grid)) in enumerate(base_models.items()):
    print(f"\n>>> {name}")
    if DO_GRIDSEARCH:
        grid = GridSearchCV(model_cls, param_grid, cv=kf, scoring='neg_root_mean_squared_error', n_jobs=GRID_JOBS)
        grid.fit(X_train_scaled, y_train)
        best_model = grid.best_estimator_
        best_params_base[name] = grid.best_params_
        print(f"  Best params: {grid.best_params_}")
    else:
        best_model = model_cls
        best_params_base[name] = {}

    # OOF predictions (cross_val_predict)
    try:
        oof = cross_val_predict(best_model, X_train_scaled, y_train, cv=kf, method='predict', n_jobs=GRID_JOBS)
    except Exception:

        oof = np.zeros(X_train_scaled.shape[0])
        for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_scaled)):
            m = clone(best_model)
            m.fit(X_train_scaled[tr_idx], y_train.iloc[tr_idx])
            oof[val_idx] = m.predict(X_train_scaled[val_idx])

    # Fit on full train and predict test
    full_model = clone(best_model)
    full_model.fit(X_train_scaled, y_train)
    test_pred = full_model.predict(X_test_scaled)

    # Save
    oof_preds[:, i] = oof
    test_preds[:, i] = test_pred
    trained_base_models[name] = full_model

    # Compute metrics and record
    train_rmse = np.sqrt(mean_squared_error(y_train, full_model.predict(X_train_scaled)))
    train_r2 = r2_score(y_train, full_model.predict(X_train_scaled))
    train_mae = mean_absolute_error(y_train, full_model.predict(X_train_scaled))
    train_mape = mape(y_train, full_model.predict(X_train_scaled))
    train_rmsre = rmsre(y_train, full_model.predict(X_train_scaled))

    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_r2 = r2_score(y_test, test_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    test_mape = mape(y_test, test_pred)
    test_rmsre = rmsre(y_test, test_pred)

    base_results.append({
        'Model': name,
        'Train_RMSE': train_rmse,
        'Train_R2': train_r2,
        'Train_MAE': train_mae,
        'Train_MAPE': train_mape,
        'Train_RMSRE': train_rmsre,
        'Test_RMSE': test_rmse,
        'Test_R2': test_r2,
        'Test_MAE': test_mae,
        'Test_MAPE': test_mape,
        'Test_RMSRE': test_rmsre,
        'Best_Params': best_params_base.get(name, {})
    })

    # Feature importance plot if available
    if hasattr(full_model, 'feature_importances_'):
        fig, ax = plt.subplots(figsize=(8,6))
        feat_imp = pd.Series(full_model.feature_importances_, index=features).nlargest(15)
        feat_imp[::-1].plot(kind='barh', ax=ax)
        ax.set_title(f"{name} - Feature importance")
        ax.set_xlabel("Importance")
        save_fig(fig, os.path.join(OUTPUT_DIR, f"{name}_feature_importance.png"))

    # Generate SHAP for base model
    try:
        print("  Generating SHAP for base model...")
        # choose explainer
        if name in ['DecisionTree', 'RandomForest', 'GradientBoosting', 'XGBoost']:
            expl = shap.TreeExplainer(full_model)
            shap_values = expl.shap_values(X_train_scaled)
            # Summary
            #fig = plt.figure(figsize=(10,8))
            shap.summary_plot(shap_values, X_train, feature_names=features, show=False)
            fig = plt.gcf()
            save_fig(fig, os.path.join(OUTPUT_DIR, f"SHAP_summary_{name}.png"))
            # Bar
            #fig = plt.figure(figsize=(10,6))
            shap.summary_plot(shap_values, X_train, feature_names=features, plot_type='bar', show=False)
            fig = plt.gcf()
            save_fig(fig, os.path.join(OUTPUT_DIR, f"SHAP_bar_{name}.png"))
        elif name in ['LinearRegression', 'Lasso']:
            expl = shap.LinearExplainer(full_model, X_train_scaled, feature_perturbation="correlation_dependent")
            shap_values = expl.shap_values(X_train_scaled)



            shap.summary_plot(shap_values, X_train, feature_names=features, show=False)
            fig = plt.gcf()  # get the figure SHAP actually drew on
            save_fig(fig, os.path.join(OUTPUT_DIR, f"SHAP_summary_{name}.png"))



            #fig = plt.figure(figsize=(10,6))
            shap.summary_plot(shap_values, X_train, feature_names=features, plot_type='bar', show=False)
            fig = plt.gcf()
            save_fig(fig, os.path.join(OUTPUT_DIR, f"SHAP_bar_{name}.png"))
        else:
            # Non-tree, non-linear simple fallback: KernelExplainer with small background
            background = shap.sample(X_train_scaled, min(100, X_train_scaled.shape[0]), random_state=RANDOM_SEED)
            expl = shap.KernelExplainer(full_model.predict, background)
            shap_values = expl.shap_values(X_train_scaled[:200]) if X_train_scaled.shape[0] > 200 else expl.shap_values(X_train_scaled)
            #fig = plt.figure(figsize=(10,8))
            shap.summary_plot(shap_values, X_train, feature_names=features, show=False)
            fig = plt.gcf()
            save_fig(fig, os.path.join(OUTPUT_DIR, f"SHAP_summary_{name}.png"))
            #fig = plt.figure(figsize=(10,6))
            shap.summary_plot(shap_values, X_train, feature_names=features, plot_type='bar', show=False)
            fig = plt.gcf()
            save_fig(fig, os.path.join(OUTPUT_DIR, f"SHAP_bar_{name}.png"))

        print(f"  SHAP saved for {name}")
    except Exception as e:
        print(f"  Could not compute SHAP for {name}: {e}")

# convert base results to df and save
base_results_df = pd.DataFrame(base_results)
base_results_df.to_csv(os.path.join(OUTPUT_DIR, "base_results.csv"), index=False)

# -----------------------
# Meta-model
# -----------------------

meta_models = {
    'LinearRegression': (LinearRegression(), {'fit_intercept': [True, False], 'positive': [True, False]}),
    'Lasso': (Lasso(), {'alpha': [0.0001, 0.001, 0.01, 0.1], 'max_iter': [1000, 5000]}),
    'XGBoost': (xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_SEED), {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.1], 'max_depth': [3, 6]})
}

meta_results = []
best_params_meta = {}
trained_meta_models = {}

print("\nTraining meta models (stacking)...")
for meta_name, (meta_model_cls, meta_params) in meta_models.items():
    print(f"\n>>> Meta: {meta_name}")
    grid = GridSearchCV(meta_model_cls, meta_params, cv=kf, scoring='neg_root_mean_squared_error', n_jobs=GRID_JOBS) if DO_GRIDSEARCH else None

    if DO_GRIDSEARCH:
        grid.fit(oof_preds, y_train)
        best_meta = grid.best_estimator_
        best_params_meta[meta_name] = grid.best_params_
        print(f"  Best params: {grid.best_params_}")
    else:
        best_meta = meta_model_cls
        best_params_meta[meta_name] = {}

    # Fit best_meta on full oof -> y_train and get predictions for oof (train) and test base preds
    best_meta.fit(oof_preds, y_train)
    meta_train_pred = best_meta.predict(oof_preds)
    meta_test_pred = best_meta.predict(test_preds)

    # rename display name for stacked XGBoost
    display_name = 'X-SEL' if meta_name == 'XGBoost' else f"Stacked_{meta_name}"

    trained_meta_models[display_name] = best_meta

    # compute metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, meta_train_pred))
    train_r2 = r2_score(y_train, meta_train_pred)
    train_mae = mean_absolute_error(y_train, meta_train_pred)
    train_mape = mape(y_train, meta_train_pred)
    train_rmsre = rmsre(y_train, meta_train_pred)

    test_rmse = np.sqrt(mean_squared_error(y_test, meta_test_pred))
    test_r2 = r2_score(y_test, meta_test_pred)
    test_mae = mean_absolute_error(y_test, meta_test_pred)
    test_mape = mape(y_test, meta_test_pred)
    test_rmsre = rmsre(y_test, meta_test_pred)

    meta_results.append({
        'Model': display_name,
        'Train_RMSE': train_rmse,
        'Train_R2': train_r2,
        'Train_MAE': train_mae,
        'Train_MAPE': train_mape,
        'Train_RMSRE': train_rmsre,
        'Test_RMSE': test_rmse,
        'Test_R2': test_r2,
        'Test_MAE': test_mae,
        'Test_MAPE': test_mape,
        'Test_RMSRE': test_rmsre,
        'Best_Params': best_params_meta.get(meta_name, {})
    })

    # Save meta predictions into test_data (if desired)
    test_df[f'{display_name}_Pred'] = meta_test_pred
    train_df[f'{display_name}_Pred'] = meta_train_pred


    # -----------------------
    try:
        print("  Generating Kernel SHAP for meta (maps original features -> meta prediction)...")
        # Prepare background (use original scaled features)

        background_size = min(100, X_train_scaled.shape[0])
        background = shap.sample(X_train_scaled, background_size, random_state=RANDOM_SEED)

        # Build wrapper prediction function
        def meta_predict_from_original(X_orig_array):
            """X_orig_array expected shape (n_samples, n_features) in scaled space
               returns array of meta predictions"""
            # compute base model predictions for each base model given original X (scaled)
            # note: base model expects scaled features
            # so we can pass X_orig_array directly (already scaled)
            if X_orig_array.ndim == 1:
                X_in = X_orig_array.reshape(1, -1)
            else:
                X_in = X_orig_array
            base_pred_list = []
            for nm in trained_base_models:
                m = trained_base_models[nm]
                base_pred_list.append(m.predict(X_in))
            # shape (n_models, n_samples) -> transpose to (n_samples, n_models)
            base_pred_matrix = np.vstack(base_pred_list).T
            # meta model expects oof-like features; predict
            return best_meta.predict(base_pred_matrix)

        # KernelExplainer. Use a small sample to compute shap values for summary.
        ke = shap.KernelExplainer(meta_predict_from_original, background)
        # select subset for explanation (max 200 samples to avoid extreme runtime)
        n_explain = min(200, X_train_scaled.shape[0])
        sample_to_explain = X_train_scaled[:n_explain]

        shap_vals_meta = ke.shap_values(sample_to_explain, nsim=200)  # nsim controls runtime; increase for better accuracy

        # Convert shap values and plot summary on original feature names
        fig = plt.figure(figsize=(12,8))
        shap.summary_plot(shap_vals_meta, pd.DataFrame(sample_to_explain, columns=features), feature_names=features, show=False)
        save_fig(fig, os.path.join(OUTPUT_DIR, f"SHAP_meta_summary_{display_name}.png"))

        # Bar plot (mean abs)
        fig = plt.figure(figsize=(12,6))
        shap.summary_plot(shap_vals_meta, pd.DataFrame(sample_to_explain, columns=features), feature_names=features, plot_type='bar', show=False)
        save_fig(fig, os.path.join(OUTPUT_DIR, f"SHAP_meta_bar_{display_name}.png"))

        print(f"  Kernel SHAP saved for meta ({display_name})")
    except Exception as e:
        print(f"  Could not compute Kernel SHAP for meta ({display_name}): {e}")

# Save meta results
meta_results_df = pd.DataFrame(meta_results)
meta_results_df.to_csv(os.path.join(OUTPUT_DIR, "meta_results.csv"), index=False)

# Save train/test predictions and results
train_df.to_excel(os.path.join(OUTPUT_DIR, "final_train_predictions.xlsx"), index=False)
test_df.to_excel(os.path.join(OUTPUT_DIR, "final_test_predictions.xlsx"), index=False)
base_results_df.to_csv(os.path.join(OUTPUT_DIR, "final_base_results.csv"), index=False)
meta_results_df.to_csv(os.path.join(OUTPUT_DIR, "final_meta_results.csv"), index=False)

# -----------------------
# Print summary tables
# -----------------------
print("\nBase Models Performance:")
print(tabulate(base_results_df, headers='keys', tablefmt='fancy_grid', showindex=False, floatfmt=".4f"))

print("\nMeta Models Performance:")
print(tabulate(meta_results_df, headers='keys', tablefmt='fancy_grid', showindex=False, floatfmt=".4f"))

print("\nAll done. Outputs saved in:", OUTPUT_DIR)


Loading data...
Features (6): ['aₓ', 'aᵧ', 'az', 'ωₓ', 'ωᵧ', 'ωz']

Training base models...

>>> LinearRegression
  Best params: {'fit_intercept': True, 'positive': False}
  Generating SHAP for base model...


Estimating transforms:   0%|          | 0/1000 [00:00<?, ?it/s]

  SHAP saved for LinearRegression

>>> DecisionTree
  Best params: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5}
  Generating SHAP for base model...
  SHAP saved for DecisionTree

>>> RandomForest
  Best params: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
  Generating SHAP for base model...
  SHAP saved for RandomForest

>>> GradientBoosting
  Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  Generating SHAP for base model...
  SHAP saved for GradientBoosting

>>> SVR
  Best params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
  Generating SHAP for base model...


  0%|          | 0/200 [00:00<?, ?it/s]

  Could not compute SHAP for SVR: Feature and SHAP matrices must have the same number of rows!

>>> kNN
  Best params: {'n_neighbors': 5, 'weights': 'uniform'}
  Generating SHAP for base model...


...

  Kernel SHAP saved for meta (Stacked_LinearRegression)

>>> Meta: Lasso
  Best params: {'alpha': 0.1, 'max_iter': 1000}
  Generating Kernel SHAP for meta (maps original features -> meta prediction)...


  0%|          | 0/200 [00:00<?, ?it/s]

  Kernel SHAP saved for meta (Stacked_Lasso)

>>> Meta: XGBoost
  Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}
  Generating Kernel SHAP for meta (maps original features -> meta prediction)...


  0%|          | 0/200 [00:00<?, ?it/s]

  Kernel SHAP saved for meta (X-SEL)

Base Models Performance:
╒══════════════════╤══════════════╤════════════╤═════════════╤══════════════╤═══════════════╤═════════════╤═══════════╤════════════╤═════════════╤══════════════╤═══════════════════════════════════════════════════════════════════════════════════════╕
│ Model            │   Train_RMSE │   Train_R2 │   Train_MAE │   Train_MAPE │   Train_RMSRE │   Test_RMSE │   Test_R2 │   Test_MAE │   Test_MAPE │   Test_RMSRE │ Best_Params                                                                           │
╞══════════════════╪══════════════╪════════════╪═════════════╪══════════════╪═══════════════╪═════════════╪═══════════╪════════════╪═════════════╪══════════════╪═══════════════════════════════════════════════════════════════════════════════════════╡
│ LinearRegression │       4.4615 │     0.9474 │      3.1847 │     294.3514 │        5.8716 │      2.8452 │ -172.9840 │     2.2075 │      5.1663 │       0.0666 │ {'fit_intercept': True, '